# Lezione 2. Il dilemma dei bandit: esplorazione e sfruttamento

**DIH DATAlife — Corso di Apprendimento per Rinforzo da zero**

> Traccia della lezione: usa questo notebook dal vivo. Ogni cella di codice è piccola e si esegue passo dopo passo, così puoi fermarti, commentare il risultato e rispondere alle domande prima di andare avanti.

## Cosa vedremo oggi

- Il dilemma tra esplorare e sfruttare, con esempi quotidiani
- Il problema del bandit a più bracci
- Come stimare il valore di un'azione
- La strategia epsilon-greedy (ε-greedy)
- La strategia Upper Confidence Bound (UCB)
- Un assaggio di Thompson Sampling
- Codice dal vivo: epsilon-greedy contro UCB, con un grafico di confronto


## Uno. Il dilemma tra esplorazione e sfruttamento

> Traccia: apri la lezione con la domanda del ristorante. Lascia che il gruppo risponda prima di continuare.

Immagina di trasferirti in una città nuova. Hai un ristorante che ti piace a cinque minuti da casa: sai cosa mangerai e sai che ti piacerà. Ma, due strade più in là, c'è un locale che non hai mai provato. Torni all'opzione sicura o rischi di scoprire qualcosa di meglio?

Questa domanda così quotidiana è lo stesso problema che affronta un'équipe medica quando sceglie tra un trattamento consolidato e uno sperimentale, o una piattaforma digitale quando decide quale pubblicità mostrarti. In tutti e tre i casi c'è una tensione tra due spinte:

- **Sfruttare** ciò che sai già funzionare
- **Esplorare** per capire se esiste qualcosa di meglio

Nell'Apprendimento per Rinforzo (RL, dall'inglese *Reinforcement Learning*), questa tensione si chiama **dilemma tra esplorazione e sfruttamento** (*exploration-exploitation dilemma*) ed è uno degli aspetti più importanti dell'intero campo.

Ricorda gli elementi base dell'RL della lezione 1: un agente che agisce, un ambiente che risponde, una politica che decide cosa fare e una ricompensa che misura se quella decisione è stata buona. Oggi vedrai come un agente decide, passo dopo passo, se ripetere ciò che conosce o provare qualcosa di nuovo, usando la versione più semplice possibile di un problema decisionale: il **bandit a più bracci**.


## Due. Il problema del bandit a più bracci

> Traccia: se hai tempo, disegna alla lavagna una fila di slot machine. È un'immagine che resta impressa.

In un casinò, una slot machine si chiama in inglese "one-armed bandit" (bandito a un braccio), perché ha una sola leva e "ti ruba" il denaro poco alla volta. Ora immagina una fila di più di queste macchine, una accanto all'altra. Ogni macchina paga premi diversi: alcune pagano molto e spesso; altre quasi mai. Il problema è che non sai quale sia quale. L'unico modo per scoprirlo è tirare le leve e osservare cosa succede. Questo scenario completo si chiama **bandit a più bracci** (*multi-armed bandit*) e dà il nome a questa famiglia di problemi.

Punti chiave:

- Ogni leva rappresenta un'azione possibile
- Ogni azione ha un **valore vero**: la ricompensa media che otterresti scegliendola infinite volte
- Quel valore esiste, ma tu non lo conosci; puoi solo stimarlo tirando la leva
- Ogni macchina distribuisce i premi secondo la stessa distribuzione fissa: non cambia nel tempo né reagisce alle tue decisioni precedenti
- Per questo è un problema "senza stato": l'unica cosa che cambia mentre giochi è la tua conoscenza, non l'ambiente

### Il topo e le due leve

Pensa a un topo da laboratorio che può scegliere tra due leve, una su sfondo nero e una su sfondo bianco. Un giorno la leva nera dà del formaggio; un altro giorno dà una scossa. Se il topo si guidasse solo dall'ultimo risultato, o da una sola brutta esperienza, potrebbe abbandonare per sempre una leva che, in media, è la scelta migliore.

Lo stesso accade a qualsiasi agente che decide in modo troppo affrettato: **una cattiva serie iniziale può convincerlo a evitare la migliore azione possibile per il resto del gioco**.


## Tre. Stimare il valore di ogni azione

> Traccia: chiedi al gruppo come calcolerebbe il valore di una leva se potesse annotare i risultati su un taccuino. Di solito esce la media.

Se non conosci il valore reale di una leva, la cosa ragionevole è stimarlo a partire da ciò che hai osservato. Il modo più diretto è calcolare la media delle ricompense ricevute ogni volta che hai scelto quell'azione. Questa stima si chiama **valore di azione** e si aggiorna ogni volta che provi di nuovo la stessa leva.

Non serve conservare tutto lo storico delle ricompense per calcolare quella media. Puoi aggiornarla in modo **incrementale**: ogni volta che osservi una nuova ricompensa, correggi la stima precedente un po' verso quel nuovo valore, con un peso che diminuisce quanto più spesso hai provato quell'azione:

$$Q_{n+1} = Q_n + \frac{1}{n}(R_n - Q_n)$$

All'inizio, ogni nuova osservazione cambia molto la tua stima; col tempo, ogni osservazione aggiuntiva pesa di meno e la stima si stabilizza vicino al valore reale.

**Variante utile:** invece di dare lo stesso peso a tutte le osservazioni passate, puoi usare un tasso di apprendimento fisso (un *learning rate* costante) che dia più peso alle ricompense recenti. È particolarmente utile in ambienti non stazionari, dove una macchina che oggi paga bene domani potrebbe pagare peggio.


## Quattro. Epsilon-greedy: esplorare un po', sfruttare molto

> Traccia: chiedi a qualcuno del gruppo di proporre un valore di ε e calcola ad alta voce quale percentuale di volte esplorerebbe.

La strategia più semplice per risolvere il dilemma si chiama **epsilon-greedy** (ε-greedy). Funziona così:

- Definisci un numero piccolo, epsilon (ε), compreso tra zero e uno
- A ogni decisione, con probabilità ε scegli un'azione a caso (**esplori**)
- Con probabilità 1−ε scegli l'azione che finora sembra migliore secondo le tue stime (**sfrutti**)

Un dettaglio curioso: poiché anche la scelta casuale può coincidere con la migliore azione nota, la probabilità reale di sfruttare è leggermente maggiore di 1−ε.

Torniamo all'esempio del ristorante. Se il tuo ε è 0,1, allora il 90% delle volte vai al tuo posto di fiducia e il 10% delle volte provi qualcosa di diverso, scelto a caso tra tutte le opzioni disponibili.

Un ε fisso garantisce che continui a esplorare per sempre: col tempo finirai per provare tutte le azioni abbastanza volte da conoscerne il valore reale con buona precisione. Ma ha un costo: **continuerai a commettere la stessa percentuale di errori** anche dopo aver imparato qual è la scelta migliore. Per questo un miglioramento frequente è far diminuire ε nel tempo. Questa strategia è stata usata, per esempio, in Deep Q-Network (DQN), uno dei sistemi che hanno imparato a giocare ai videogiochi Atari partendo direttamente dai pixel dello schermo.


## Cinque. Upper Confidence Bound: l'ottimismo di fronte all'incertezza

> Traccia: introduci la frase "ottimismo di fronte all'incertezza" e chiedi di collegarla alla vita reale (provare un nuovo fornitore, una ricetta nuova).

Epsilon-greedy ha un limite: quando esplora, sceglie a caso tra tutte le azioni, senza distinguere tra una leva che conosci bene e un'altra che hai appena provato. **Upper Confidence Bound (UCB)** risolve questo problema in modo più intelligente.

La logica di UCB: invece di guardare solo la tua stima attuale del valore di ogni azione, le aggiungi un **bonus di incertezza**:

- È grande per le azioni che hai provato poche volte (concedi loro il beneficio del dubbio)
- È piccolo per quelle che hai provato molte volte (hai già una stima affidabile)
- A ogni passo scegli l'azione con la somma più alta tra **valore stimato + bonus**

all'inizio, dato che non hai provato nessuna leva, il bonus delle azioni non provate è enorme, quindi UCB ti obbliga a provare ogni azione almeno una volta prima di iniziare a confrontarle. Man mano che giochi, il bonus di ogni leva si riduce con il crescere delle volte che la provi, seguendo grosso modo il ritmo della radice quadrata del numero di tentativi. Così l'algoritmo passa in modo naturale da una fase di forte esplorazione a una fase concentrata sullo sfruttamento di ciò che già sa.

Esistono varianti di UCB per scenari diversi: UCB1-Tuned adatta il bonus in base a quanto variano le ricompense osservate; UCB Scontato dà più peso alle osservazioni recenti; UCB Contestuale aggiunge informazioni sulla situazione (per esempio, le caratteristiche dell'utente in un sistema di raccomandazione).


## Sei. Thompson Sampling: un'alternativa bayesiana

> Traccia: se hai poco tempo, questa sezione si può riassumere in due minuti; il focus della pratica di oggi è epsilon-greedy e UCB.

Esiste una terza famiglia di strategie, di impostazione bayesiana: **Thompson Sampling** (chiamato anche campionamento di Thompson o bandit bayesiano). Invece di conservare un'unica stima per azione, mantiene una **distribuzione di probabilità completa** sul valore di ogni azione: una sorta di "credenza" che riflette sia la tua migliore stima sia la tua incertezza su di essa.

Ad ogni passo:

1. Per ogni azione, ottieni un valore campione dalla sua distribuzione di credenza attuale
2. Scegli l'azione con il campione più alto
3. Osserva la ricompensa reale e aggiorna la credenza di quell'azione

Quando un'azione ha una distribuzione molto incerta, i suoi campioni variano molto e a volte escono valori alti: questa è esplorazione. Quando una distribuzione si è consolidata attorno a un valore alto, i suoi campioni sono quasi sempre alti e coerenti: questo è sfruttamento. Tutto avviene in modo naturale, senza bisogno di un parametro come ε né di una formula esplicita di bonus.

Per ricompense binarie (successo o fallimento), Thompson Sampling usa di solito la distribuzione Beta: si parte da una distribuzione neutra e, ogni volta che osservi un successo, si aggiorna un parametro; ogni volta che osservi un fallimento, si aggiorna l'altro.


## Sette. Codice dal vivo: epsilon-greedy contro UCB

> Traccia: da qui in poi, esegui ogni cella separatamente e commenta il risultato prima di passare alla successiva. È lo stesso codice del manuale, suddiviso in passi.

Simulerai un bandit a più bracci e confronterai visivamente come si comportano epsilon-greedy e UCB lungo molti passi. Ti servono solo `numpy` e `matplotlib`, che sono già preinstallati in Google Colab.

**Esegui ogni cella in ordine.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fijamos una semilla para que el experimento sea reproducible
np.random.seed(42)


### Passo uno: configura il bandit

Definisci quanti bracci ha il bandit, quanti passi giocherà ciascuna strategia e quante volte ripeti l'intero esperimento per attenuare l'effetto del caso.


In [ ]:
# --- Configuración del bandido multi-brazo ---
n_brazos = 10          # número de "palancas" o acciones disponibles
n_pasos = 1000          # cuántas veces va a jugar cada estrategia
n_repeticiones = 200    # repetimos el experimento varias veces para suavizar el azar

# El valor verdadero de cada brazo se genera al azar (el agente NO lo conoce)
# Estos valores representan la recompensa media real de cada acción
valores_reales = np.random.normal(loc=0.0, scale=1.0, size=n_brazos)
valores_reales


### Passo due: simula il tiro di una leva

Ogni volta che l'agente sceglie un braccio, riceve una ricompensa con rumore attorno al valore reale di quel braccio, come accadrebbe con una vera slot machine.


In [ ]:
def tirar_de_la_palanca(brazo):
    """Simula el resultado de elegir un brazo: devuelve una recompensa con ruido
    alrededor del valor verdadero de ese brazo (como en una tragaperras real)."""
    return np.random.normal(loc=valores_reales[brazo], scale=1.0)


# Prueba rápida: tira varias veces del brazo 0 y observa la variación
[round(tirar_de_la_palanca(0), 2) for _ in range(5)]


### Passo tre: implementa epsilon-greedy

Con probabilità ε esplora a caso; le altre volte sfrutta la migliore azione conosciuta finora.


In [ ]:
def ejecutar_epsilon_greedy(epsilon, n_pasos):
    """Estrategia epsilon-greedy: explora al azar con probabilidad epsilon,
    y el resto de las veces elige la acción con mejor valor estimado."""
    valores_estimados = np.zeros(n_brazos)   # Q_t(a): estimación actual de cada brazo
    conteo_intentos = np.zeros(n_brazos)     # cuántas veces se ha probado cada brazo
    recompensas = np.zeros(n_pasos)

    for t in range(n_pasos):
        if np.random.random() < epsilon:
            # Exploración: elige un brazo al azar
            brazo = np.random.randint(n_brazos)
        else:
            # Explotación: elige el brazo con mejor valor estimado hasta ahora
            brazo = np.argmax(valores_estimados)

        recompensa = tirar_de_la_palanca(brazo)
        conteo_intentos[brazo] += 1

        # Actualización incremental de la media muestral (sección tres):
        # Q_{n+1} = Q_n + (1/n) * (R_n - Q_n)
        paso = 1.0 / conteo_intentos[brazo]
        valores_estimados[brazo] += paso * (recompensa - valores_estimados[brazo])

        recompensas[t] = recompensa

    return recompensas


### Passo quattro: implementa UCB

Combina il valore stimato con un bonus di incertezza che favorisce i bracci poco provati.


In [ ]:
def ejecutar_ucb(c, n_pasos):
    """Estrategia UCB: combina el valor estimado con un bono de incertidumbre
    que favorece a los brazos poco probados."""
    valores_estimados = np.zeros(n_brazos)
    conteo_intentos = np.zeros(n_brazos)
    recompensas = np.zeros(n_pasos)

    for t in range(1, n_pasos + 1):
        # Si algún brazo aún no se ha probado, se elige primero (bono "infinito")
        brazos_sin_probar = np.where(conteo_intentos == 0)[0]
        if len(brazos_sin_probar) > 0:
            brazo = brazos_sin_probar[0]
        else:
            # UCB: valor estimado + bono de exploración que crece con log(t)
            # y se reduce con la raíz cuadrada del número de intentos del brazo
            bono = c * np.sqrt(np.log(t) / conteo_intentos)
            brazo = np.argmax(valores_estimados + bono)

        recompensa = tirar_de_la_palanca(brazo)
        conteo_intentos[brazo] += 1
        paso = 1.0 / conteo_intentos[brazo]
        valores_estimados[brazo] += paso * (recompensa - valores_estimados[brazo])

        recompensas[t - 1] = recompensa

    return recompensas


### Passo cinque: esegui entrambe le strategie molte volte

Ripeti l'esperimento 200 volte, rigenerando i valori reali a ogni ripetizione, così che la media non dipenda da un'unica configurazione di bracci.

> Traccia: questa cella richiede alcuni secondi. Approfitta per spiegare perché ripetiamo l'esperimento invece di eseguirlo una sola volta.


In [ ]:
# --- Ejecutamos ambas estrategias varias veces y promediamos ---
recompensas_epsilon = np.zeros(n_pasos)
recompensas_ucb = np.zeros(n_pasos)

for _ in range(n_repeticiones):
    # Re-generamos los valores reales en cada repetición para que el promedio
    # no dependa de una única configuración de brazos
    valores_reales = np.random.normal(loc=0.0, scale=1.0, size=n_brazos)
    recompensas_epsilon += ejecutar_epsilon_greedy(epsilon=0.1, n_pasos=n_pasos)
    recompensas_ucb += ejecutar_ucb(c=2.0, n_pasos=n_pasos)

recompensas_epsilon /= n_repeticiones
recompensas_ucb /= n_repeticiones

print("Listo: experimento completado")


### Passo sei: calcola la ricompensa media cumulata

Trasforma la ricompensa ottenuta a ogni passo in una ricompensa media cumulata, per vedere l'andamento dell'apprendimento di ciascuna strategia.


In [ ]:
# Convertimos la recompensa por paso en recompensa media acumulada
recompensa_media_acumulada_epsilon = np.cumsum(recompensas_epsilon) / np.arange(1, n_pasos + 1)
recompensa_media_acumulada_ucb = np.cumsum(recompensas_ucb) / np.arange(1, n_pasos + 1)

print("Recompensa media acumulada final, epsilon-greedy:", round(recompensa_media_acumulada_epsilon[-1], 3))
print("Recompensa media acumulada final, UCB:", round(recompensa_media_acumulada_ucb[-1], 3))


### Passo sette: grafica il confronto

> Traccia: prima di eseguire, chiedi al gruppo quale curva pensa che sarà in testa e perché.


In [ ]:
# --- Graficamos la comparación ---
plt.figure(figsize=(10, 5))
plt.plot(recompensa_media_acumulada_epsilon, label="epsilon-greedy (ε = 0.1)")
plt.plot(recompensa_media_acumulada_ucb, label="UCB (c = 2.0)")
plt.xlabel("Paso de tiempo")
plt.ylabel("Recompensa media acumulada")
plt.title("Comparación epsilon-greedy vs. UCB en un bandido de 10 brazos")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Cosa osservare nel grafico

Entrambe le curve migliorano nel tempo, ma **UCB tende a superare epsilon-greedy**: poiché distribuisce l'esplorazione in modo più intelligente (provando prima ciò che conosce meno), trova il braccio ottimo più in fretta e spreca meno tentativi sui bracci chiaramente cattivi.


## Punti chiave della lezione

- Il dilemma tra esplorazione e sfruttamento compare in ogni decisione in condizioni di incertezza, dentro e fuori dall'RL
- Il bandit a più bracci è la versione più semplice di questo problema: più azioni, ciascuna con un valore reale sconosciuto e fisso nel tempo
- Il valore di un'azione si stima con la media delle ricompense osservate, e può essere aggiornato in modo incrementale senza conservare tutto lo storico
- Epsilon-greedy esplora a caso con probabilità ε e sfrutta il resto del tempo; un ε decrescente di solito funziona meglio di uno fisso
- UCB dirige l'esplorazione verso le azioni più incerte, sommando un bonus che si riduce man mano che un'azione viene provata di più
- Thompson Sampling usa distribuzioni di probabilità complete e bilancia esplorazione e sfruttamento in modo naturale, senza parametri espliciti come ε

## Passo successivo

**Apri ora il notebook di pratica** (`clase02_practica_bandidos.ipynb`) **e completa i due esercizi guidati!** Implementerai UCB da zero e confronterai diversi valori di ε.

Nella lezione 3 lascerai il mondo "senza stato" dei bandit ed entrerai nei Processi Decisionali di Markov, dove le azioni dell'agente cambiano davvero lo stato dell'ambiente.
